# Daily Practice — 2026-11-09 — ML Testing & MLOps: A Data Validation Test Suite for an Incoming Batch

**Dataset:** [Titanic](https://github.com/mwaskom/seaborn-data) — 891 real passenger records from the
RMS Titanic (class, sex, age, fare, embarkation port, family size, survival), loaded straight from
`mwaskom/seaborn-data` on GitHub. It isn't a model-accuracy dataset here — it's standing in for a
tabular feature pipeline. Real messiness included: missing `age`/`embarked` values, mixed numeric and
categorical columns, and no unique passenger ID (so "duplicate row" really does mean "duplicate row").

**Problem statement.** You're the QA engineer embedded with an ML team that scores this kind of tabular
data daily. A new batch lands from an upstream system before every scoring run, and right now *nothing*
checks it before it hits the model — if the upstream system starts sending `pclass=4`, negative fares,
or a flood of new nulls, the model silently scores garbage. Your job is to build the validation gate that
should have existed from day one: a small test suite, but for data instead of code, that runs against
every incoming batch and fails loudly — with specifics — when the batch violates the assumptions the
model was trained on.

**What you should produce:**
1. A family of **check functions** — schema, numeric range, categorical domain, null-rate, duplicate-rate,
   and cross-field logic — each returning a structured `ValidationResult`, not just `True`/`False`.
2. A `run_validation_suite(df, checks)` function that runs a list of checks and returns a tidy report
   (one row per check: name, pass/fail, violation count, violation rate, detail).
3. A `corrupt_batch(df, rng)` function that takes a *real* clean batch and injects a labelled set of data
   quality problems (bad ages, an unexpected category, extra nulls, duplicated rows, a cross-field
   inconsistency) — this is the one place we deliberately inject synthetic noise, and only because we
   need corruption whose ground truth we control, to prove each check actually catches what it claims to.
4. A side-by-side report: the suite run on the clean batch (should pass, or fail only on things you've
   explicitly allowed for) vs. the corrupted batch (should fail, with the *right* checks failing).


## Setup

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass

pd.set_option("display.max_colwidth", 80)
RANDOM_STATE = 42


## Load the data and define the reference baseline vs. the incoming batch

We keep the columns a scoring model would actually use, split the 891 rows into a `df_reference`
chunk (the first 700 rows — stand-in for "what the pipeline looked like when the model was built and
validated") and a `df_new_batch` chunk (the remaining 191 rows — stand-in for "today's incoming data").
`df_reference` is what you calibrate your check *thresholds* against; `df_new_batch` is what you actually
validate.

In [ ]:
COLS = [
    "survived", "pclass", "sex", "age", "sibsp", "parch",
    "fare", "embarked", "class", "who", "embark_town",
]

df = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
)[COLS].copy()

df_reference = df.iloc[:700].reset_index(drop=True)
df_new_batch = df.iloc[700:].reset_index(drop=True)

print(df_reference.shape, df_new_batch.shape)
print("\nReference null rates:\n", df_reference.isna().mean().round(3))
df_new_batch.head()


## Your task

Fill in the TODOs below. Signatures and the `ValidationResult` dataclass are given — keep them as-is,
the later cells (and the solution) assume these names.

Notes on a couple of design choices, so you're not guessing:
- **Schema check works on logical "kind" (`numeric` / `categorical`), not exact pandas dtype.** Exact
  dtypes drift across pandas versions and read paths (`object` vs. a dedicated string dtype); checking
  "is this numeric or not" is what you actually care about for a model input contract.
- **Thresholds for null-rate and duplicate-rate aren't zero.** Look at `df_reference`'s own null rates
  and duplicate rate first (previous cell) and pick thresholds with headroom above that baseline — a
  good check should pass the reference data trivially and only fire on a *real* shift.

In [ ]:
@dataclass
class ValidationResult:
    check_name: str
    passed: bool
    n_violations: int
    n_rows: int
    detail: str = ""

    @property
    def violation_rate(self):
        return self.n_violations / self.n_rows if self.n_rows else 0.0


def check_schema(df, expected_columns, expected_kinds):
    '''expected_kinds: dict[col] -> 'numeric' or 'categorical'.
    Fail if any expected column is missing, any extra column is present,
    or any column's numeric-ness doesn't match expected_kinds.
    '''
    # TODO
    raise NotImplementedError


def check_value_range(df, column, min_val, max_val):
    '''Fail if any non-null value in `column` falls outside [min_val, max_val].'''
    # TODO
    raise NotImplementedError


def check_categorical_domain(df, column, allowed_values):
    '''Fail if any non-null value in `column` is not in allowed_values.'''
    # TODO
    raise NotImplementedError


def check_null_rate(df, column, max_null_rate):
    '''Fail if the fraction of nulls in `column` exceeds max_null_rate.'''
    # TODO
    raise NotImplementedError


def check_duplicates(df, max_duplicate_rate):
    '''Fail if the fraction of fully-duplicated rows exceeds max_duplicate_rate.'''
    # TODO
    raise NotImplementedError


def check_cross_field_logic(df):
    '''Fail if: survived not in {0, 1}; OR fare < 0; OR (who == 'child' AND age >= 18).'''
    # TODO
    raise NotImplementedError


def run_validation_suite(df, checks):
    '''checks: list of zero-arg callables, each returning a ValidationResult.
    Return a tidy pd.DataFrame report: one row per check, columns
    ['check', 'passed', 'n_violations', 'n_rows', 'violation_rate', 'detail'].
    '''
    # TODO
    raise NotImplementedError


### Configure the checks and run the suite on the clean incoming batch

Build the list of checks against `df_new_batch` (pick your own thresholds using the guidance above —
there's no single "correct" number, but they should be defensible from `df_reference`'s stats). Then run
`run_validation_suite` and display the report. On the real, un-corrupted batch you should see all checks
pass (small, expected null rates and duplicate rates are *not* failures if your thresholds account for
them).

In [ ]:
def build_checks(df):
    # TODO: return a list of zero-arg lambdas, one per check you want to run,
    # e.g. lambda: check_value_range(df, "age", 0, 100)
    raise NotImplementedError


# TODO: run the suite on df_new_batch and display the report


## Now corrupt a copy of the batch

Write `corrupt_batch(df, rng)`. It should return a **copy** of `df` with all of the following injected
(use `rng`, a `numpy.random.Generator`, for every random choice so the corruption is reproducible):

1. Set `age` to a negative value for a handful of rows (~5% of rows).
2. Set `sex` to the out-of-domain value `"unknown"` for a few rows (~3%).
3. Set `pclass` to the out-of-domain value `4` for a few rows (~2–3%).
4. Duplicate roughly 10% of rows (append them back onto the DataFrame).
5. Null out `embarked` for an extra ~12% of rows, on top of whatever's already null.
6. For a few rows where `who == "child"`, set `age` to `35` (a direct cross-field contradiction).
7. Set `fare` to `-10.0` for a couple of rows (~2%).

This is the one function in the notebook that touches synthetic/injected values rather than the raw
data — deliberately, because proving a check catches a problem requires controlling the ground truth of
that problem. Everything your suite validates is still real Titanic data; only the specific corrupted
cells are synthetic.

In [ ]:
def corrupt_batch(df, rng):
    # TODO
    raise NotImplementedError


# TODO: corrupt df_new_batch with a seeded rng, run the suite on the result,
# and display the report next to the clean one. Which checks flip from pass to fail?


### Write-up

TODO: in 3–5 sentences — which checks actually caught the corruption and which stayed green despite the
extra noise (and why that's correct, not a bug); which single check would you put *first* in a
production gate if you had to fail fast before running the (slower) others; and what a purely rule-based
validation suite like this one structurally cannot catch, that a statistical drift detector (e.g. PSI on
`fare` or `age`) would.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
@dataclass
class ValidationResult:
    check_name: str
    passed: bool
    n_violations: int
    n_rows: int
    detail: str = ""

    @property
    def violation_rate(self):
        return self.n_violations / self.n_rows if self.n_rows else 0.0


def check_schema(df, expected_columns, expected_kinds):
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]
    kind_mismatches = []
    for col, kind in expected_kinds.items():
        if col not in df.columns:
            continue
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        if kind == "numeric" and not is_numeric:
            kind_mismatches.append(col)
        elif kind == "categorical" and is_numeric:
            kind_mismatches.append(col)
    passed = not missing and not extra and not kind_mismatches
    detail = f"missing={missing}, extra={extra}, kind_mismatches={kind_mismatches}"
    return ValidationResult("schema", passed, len(missing) + len(extra) + len(kind_mismatches), 1, detail)


def check_value_range(df, column, min_val, max_val):
    s = df[column].dropna()
    viol = s[(s < min_val) | (s > max_val)]
    passed = len(viol) == 0
    return ValidationResult(f"range:{column}", passed, len(viol), len(df), f"outside [{min_val}, {max_val}]")


def check_categorical_domain(df, column, allowed_values):
    s = df[column].dropna()
    viol = s[~s.isin(allowed_values)]
    passed = len(viol) == 0
    detail = f"unexpected values: {sorted(viol.unique().tolist())[:5]}"
    return ValidationResult(f"domain:{column}", passed, len(viol), len(df), detail)


def check_null_rate(df, column, max_null_rate):
    null_rate = df[column].isna().mean()
    passed = null_rate <= max_null_rate
    n_null = int(df[column].isna().sum())
    detail = f"null_rate={null_rate:.3f} (max {max_null_rate})"
    return ValidationResult(f"nulls:{column}", passed, n_null, len(df), detail)


def check_duplicates(df, max_duplicate_rate):
    dup_mask = df.duplicated()
    dup_rate = dup_mask.mean()
    passed = dup_rate <= max_duplicate_rate
    detail = f"dup_rate={dup_rate:.3f} (max {max_duplicate_rate})"
    return ValidationResult("duplicates", passed, int(dup_mask.sum()), len(df), detail)


def check_cross_field_logic(df):
    violations = pd.Series(False, index=df.index)
    violations |= ~df["survived"].isin([0, 1])
    violations |= df["fare"] < 0
    violations |= (df["who"] == "child") & (df["age"] >= 18)
    passed = not violations.any()
    return ValidationResult("cross_field_logic", passed, int(violations.sum()), len(df))


def run_validation_suite(df, checks):
    results = [check() for check in checks]
    return pd.DataFrame([
        {
            "check": r.check_name,
            "passed": r.passed,
            "n_violations": r.n_violations,
            "n_rows": r.n_rows,
            "violation_rate": round(r.violation_rate, 4),
            "detail": r.detail,
        }
        for r in results
    ])


def build_checks(df):
    return [
        lambda: check_schema(df, COLS, {
            "survived": "numeric", "pclass": "numeric", "sex": "categorical", "age": "numeric",
            "sibsp": "numeric", "parch": "numeric", "fare": "numeric", "embarked": "categorical",
            "class": "categorical", "who": "categorical", "embark_town": "categorical",
        }),
        lambda: check_value_range(df, "age", 0, 100),
        lambda: check_value_range(df, "fare", 0, 600),
        lambda: check_value_range(df, "sibsp", 0, 10),
        lambda: check_value_range(df, "parch", 0, 10),
        lambda: check_categorical_domain(df, "sex", ["male", "female"]),
        lambda: check_categorical_domain(df, "pclass", [1, 2, 3]),
        lambda: check_categorical_domain(df, "embarked", ["C", "Q", "S"]),
        lambda: check_categorical_domain(df, "class", ["First", "Second", "Third"]),
        lambda: check_categorical_domain(df, "who", ["man", "woman", "child"]),
        lambda: check_null_rate(df, "age", 0.30),          # reference baseline ~0.21
        lambda: check_null_rate(df, "embarked", 0.02),     # reference baseline ~0.001
        lambda: check_null_rate(df, "embark_town", 0.02),
        lambda: check_null_rate(df, "sex", 0.01),
        lambda: check_duplicates(df, 0.08),                # reference baseline ~0.12 (no unique ID!)
        lambda: check_cross_field_logic(df),
    ]


report_clean = run_validation_suite(df_new_batch, build_checks(df_new_batch))
print(report_clean.to_string())
print("\nAll passed:", report_clean["passed"].all())
```

On `df_new_batch` every check passes: its null rates (~17% for `age`, <1% for `embarked`) and duplicate
rate (~5%, unsurprising with only 11 low-cardinality columns and no passenger ID) sit comfortably under
the thresholds calibrated from `df_reference`.

```python
def corrupt_batch(df, rng):
    corrupted = df.copy()
    n = len(corrupted)

    neg_age_idx = rng.choice(corrupted.index, size=max(1, n // 20), replace=False)
    corrupted.loc[neg_age_idx, "age"] = -1 * corrupted.loc[neg_age_idx, "age"].abs().fillna(5)

    bad_sex_idx = rng.choice(corrupted.index, size=max(1, n // 30), replace=False)
    corrupted.loc[bad_sex_idx, "sex"] = "unknown"

    bad_class_idx = rng.choice(corrupted.index, size=max(1, n // 40), replace=False)
    corrupted.loc[bad_class_idx, "pclass"] = 4

    dup_idx = rng.choice(corrupted.index, size=max(1, n // 10), replace=False)
    corrupted = pd.concat([corrupted, corrupted.loc[dup_idx]], ignore_index=True)

    extra_null_idx = rng.choice(corrupted.index, size=max(1, len(corrupted) // 8), replace=False)
    corrupted.loc[extra_null_idx, "embarked"] = np.nan

    child_idx = corrupted.index[corrupted["who"] == "child"]
    if len(child_idx) > 0:
        flip_idx = rng.choice(child_idx, size=min(3, len(child_idx)), replace=False)
        corrupted.loc[flip_idx, "age"] = 35

    neg_fare_idx = rng.choice(corrupted.index, size=max(1, n // 50), replace=False)
    corrupted.loc[neg_fare_idx, "fare"] = -10.0

    return corrupted


rng = np.random.default_rng(RANDOM_STATE)
df_corrupted = corrupt_batch(df_new_batch, rng)
report_corrupt = run_validation_suite(df_corrupted, build_checks(df_corrupted))
print(report_corrupt.to_string())
print("\nAll passed:", report_corrupt["passed"].all())
print("Checks failed:", report_corrupt[~report_corrupt["passed"]]["check"].tolist())
```

On the corrupted batch, 7 of 16 checks flip to failing: `range:age`, `range:fare`, `domain:sex`,
`domain:pclass`, `nulls:embarked`, `duplicates`, and `cross_field_logic`. `schema` still passes
(corruption changed values, not columns or dtypes) and the untouched checks (`range:sibsp`,
`range:parch`, `domain:embarked`, `domain:class`, `domain:who`, `nulls:age`, `nulls:embark_town`,
`nulls:sex`) correctly stay green — they weren't the targets of any injected corruption.

**Write-up pointers:** `check_schema` is the cheapest check and the one most worth running first in a
real gate — a schema break means every other check's numbers are meaningless anyway (e.g. a column that
is suddenly a string of digits will blow through a numeric range check for the wrong reason). What this
rule-based suite structurally can't catch is a *distributional* shift that stays inside every rule — e.g.
if next week's `fare` values are all technically within `[0, 600]` and `pclass` is still `{1,2,3}`, but
the mean fare has quietly doubled, every check above passes while the model's input distribution has
clearly moved. That's exactly the gap a statistical drift detector (PSI, KS-test) is built to close, and
why a real pipeline runs both: rule-based validation for "is this even a legal value," drift detection for
"does this still look like what the model was trained on."

</details>